# 🧠 Aula 09: Avaliação Crítica de Modelos de Deep Learning e Métricas sob Desbalanceamento

**Curso:** Redes Neurais Profundas — Graduação Faculdade Infnet  
**Conexão com a Avaliação:** Gabarito conceitual e prático completo do **Estudo de Caso 1 (CNN para Triagem de Malária)**.  

---

### 🎯 Objetivos de Aprendizagem
1. **Desmascarar a Falácia da Acurácia**: Compreender por que em distribuições desbalanceadas (ex: 85% negativos / 15% positivos), acurácias elevadas (como 88,3%) ocultam modelos clinicamente desastrosos.
2. **Dominar Métricas de Risco Clínico**: Calcular e interpretar Sensibilidade (*Recall*), Especificidade, *Precision-Recall AUC* (PR-AUC) e *F-Beta Score* ($\beta = 2.0$).
3. **Mitigar Desbalanceamento no Treino**: Configurar ponderação de classes em .
4. **Ajuste de Limiar de Decisão (*Threshold Tuning*)**: Ajustar o corte de probabilidades para garantir Sensibilidade $\ge 95\%$ conforme exigido pela OMS.
5. **Calibração de Probabilidades & ECE**: Identificar redes superconfiantes e aplicar *Temperature Scaling*.


## 1. Configuração do Ambiente e Importação de Bibliotecas

Instalamos o `kagglehub` e importamos os módulos essenciais do **PyTorch**, **Scikit-Learn**, **Pandas**, **NumPy**, **Matplotlib** e **Seaborn**.


In [ ]:
# Instalação do kagglehub para download direto do dataset
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, precision_recall_curve, auc, roc_curve, confusion_matrix,
    classification_report, cohen_kappa_score, mean_absolute_error
)

# Configurações visuais elegantes
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Dispositivo de Execução: {device}")


## 2. Carregamento e Exploração dos Dados (Women's Clothing Reviews)

Baixamos o dataset diretamente do Kaggle via `kagglehub` e exploramos a distribuição dos nossos dois alvos de interesse: `Recommended IND` (Binário) e `Rating` (Multiclasse / Ordinal).


In [ ]:
import kagglehub
import os

# Download do dataset oficial via kagglehub
path = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")

# O 'path' é um diretório. Vamos listar os arquivos e encontrar o CSV.
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

df = pd.read_csv(full_path)
print(f"📊 Dimensões do Dataset: {df.shape}")
print(df[['Review Text', 'Rating', 'Recommended IND']].head())

# 1. Distribuição do Alvo Binário (Recommended IND)
dist_bin = df['Recommended IND'].value_counts(normalize=True) * 100
print(f"\n🎯 Distribuição do Alvo Binário (Recommended IND):")
print(f"  • Classe 1 (Recomendado):     {dist_bin.get(1, 0):.2f}% ({df['Recommended IND'].value_counts().get(1, 0)} amostras)")
print(f"  • Classe 0 (Não Recomendado): {dist_bin.get(0, 0):.2f}% ({df['Recommended IND'].value_counts().get(0, 0)} amostras)")

# 2. Distribuição do Alvo Multiclasse (Rating 1 a 5 estrelas)
dist_multi = df['Rating'].value_counts(normalize=True).sort_index() * 100
print(f"\n⭐ Distribuição do Alvo Multiclasse (Rating):")
for star, pct in dist_multi.items():
    print(f"  • {star} Estrela(s): {pct:.2f}% ({df['Rating'].value_counts()[star]} amostras)")

# 3. Análise de Vazamento: Relação cruzada entre Rating e Recomendação (% por estrela)
print("\n🔍 Relação cruzada entre Rating e Recomendação (% por estrela):")
crosstab = pd.crosstab(df['Rating'], df['Recommended IND'], normalize='index') * 100
crosstab.columns = ['% Não Recomendado', '% Recomendado']
print(crosstab.round(2))
print("\n👉 Rating 5 é 99.8% recomendado e Rating 1 é 98.1% não-recomendado. Por isso, Rating NÃO entra como feature!")
print("👉 Positive Feedback Count é um dado a posteriori (criado depois da publicação) e também NÃO entra como feature!")


## 3. Pré-Processamento e Particionamento Seguro

### 🛡️ Pipeline 100% Baseado em Texto:
1. **Entrada ($X$):** Texto da avaliação (`Review Text`) vetorizado via **TF-IDF** com 500 termos mais relevantes.
2. **Alvos ($y$):**
   - $y_{\text{bin}}$: `Recommended IND` ($0$ ou $1$, `FloatTensor` para `BCEWithLogitsLoss`).
   - $y_{\text{multi}}$: `Rating` $- 1$ (classes $0, 1, 2, 3, 4$, `LongTensor` para `CrossEntropyLoss`).
3. O vocabulário **TF-IDF** é ajustado (**`fit`**) **estritamente nos dados de Treino** e apenas aplicado (**`transform`**) na Validação e Teste, impedindo que palavras do teste vazem para o treino!


In [ ]:
# 1. Tratamento de textos nulos
df['Review Text'] = df['Review Text'].fillna('').astype(str)

# 2. Definição de Atributos de Texto (X) e Alvos (y_bin e y_multi)
X_text = df['Review Text'].values
y_bin = df['Recommended IND'].values
y_multi = (df['Rating'] - 1).values # Classes 0..4 para PyTorch CrossEntropy

# 3. Divisão Estratificada (70% Treino, 15% Validação, 15% Teste)
X_tr_val_text, X_test_text, y_tr_val_bin, y_test_bin, y_tr_val_multi, y_test_multi = train_test_split(
    X_text, y_bin, y_multi, test_size=0.15, random_state=42, stratify=y_multi
)

X_train_text, X_val_text, y_train_bin, y_val_bin, y_train_multi, y_val_multi = train_test_split(
    X_tr_val_text, y_tr_val_bin, y_tr_val_multi, test_size=0.1765, random_state=42, stratify=y_tr_val_multi
)

print(f"📦 Volume de Amostras -> Treino: {len(y_train_bin)} ({len(y_train_bin)/len(df):.1%}) | Validação: {len(y_val_bin)} ({len(y_val_bin)/len(df):.1%}) | Teste: {len(y_test_bin)} ({len(y_test_bin)/len(df):.1%})")

# 4. TF-IDF ajustado estritamente no Treino
tfidf = TfidfVectorizer(max_features=500, stop_words='english', sublinear_tf=True)
X_train_feat = tfidf.fit_transform(X_train_text).toarray()
X_val_feat = tfidf.transform(X_val_text).toarray()
X_test_feat = tfidf.transform(X_test_text).toarray()

input_dim = X_train_feat.shape[1]
print(f"✅ Dimensão final dos atributos de texto (TF-IDF): {input_dim}")

# 5. Criar DataLoaders do PyTorch para Ambos os Cenários
# DataLoaders Binários
train_loader_bin = DataLoader(TensorDataset(torch.FloatTensor(X_train_feat), torch.FloatTensor(y_train_bin)), batch_size=64, shuffle=True)
val_loader_bin = DataLoader(TensorDataset(torch.FloatTensor(X_val_feat), torch.FloatTensor(y_val_bin)), batch_size=128, shuffle=False)
test_loader_bin = DataLoader(TensorDataset(torch.FloatTensor(X_test_feat), torch.FloatTensor(y_test_bin)), batch_size=128, shuffle=False)

# DataLoaders Multiclasse (alvos tipo LongTensor para CrossEntropyLoss)
train_loader_multi = DataLoader(TensorDataset(torch.FloatTensor(X_train_feat), torch.LongTensor(y_train_multi)), batch_size=64, shuffle=True)
val_loader_multi = DataLoader(TensorDataset(torch.FloatTensor(X_val_feat), torch.LongTensor(y_val_multi)), batch_size=128, shuffle=False)
test_loader_multi = DataLoader(TensorDataset(torch.FloatTensor(X_test_feat), torch.LongTensor(y_test_multi)), batch_size=128, shuffle=False)


## 4. PARTE 1 — Classificação Binária (`Recommended IND`)

Nosso primeiro objetivo é prever se uma avaliação é positiva (**1 = Recomendado**) ou negativa (**0 = Não Recomendado**).
- **Função de Perda:** `nn.BCEWithLogitsLoss()` (estável numericamente com Log-Sum-Exp).
- **Otimizador:** `optim.AdamW` com Weight Decay para regularização.
- **Métricas:** Matriz de Confusão 2x2, Acurácia, Precisão, Recall, Especificidade, $F_1$, $F_2$, $F_{0.5}$ e Curvas ROC vs PR.


In [ ]:
class BinaryReviewClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1) # Saída de 1 logit linear
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# Instanciar e treinar modelo binário
set_seed(42)
model_bin = BinaryReviewClassifier(input_dim).to(device)
criterion_bin = nn.BCEWithLogitsLoss()
optimizer_bin = optim.AdamW(model_bin.parameters(), lr=1e-3, weight_decay=1e-2)

epochs = 8
print("🔄 Treinando Modelo Binário...")
for epoch in range(1, epochs + 1):
    model_bin.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader_bin:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer_bin.zero_grad()
        logits = model_bin(batch_x)
        loss = criterion_bin(logits, batch_y)
        loss.backward()
        optimizer_bin.step()
        total_loss += loss.item() * len(batch_y)
    
    avg_loss = total_loss / len(train_loader_bin.dataset)
    print(f"  • Época {epoch:02d}/{epochs:02d} | Loss Médio de Treino: {avg_loss:.4f}")

print("✅ Treinamento Binário Concluído!")


### 4.1 Métricas da Matriz de Confusão Binária (2x2)

A **Matriz de Confusão 2x2** divide as predições em:
- **TP (Verdadeiro Positivo):** Recomendados que o modelo acertou.
- **TN (Verdadeiro Negativo):** Não-recomendados que o modelo identificou corretamente.
- **FP (Falso Positivo):** Alarme falso (disse que recomendava, mas era negativo).
- **FN (Falso Negativo):** Omissão perigosa (disse que não recomendava, mas o cliente gostou).

| Métrica | Fórmula | Interpretação Prática |
| :--- | :---: | :--- |
| **Acurácia** | $\frac{TP + TN}{TP + TN + FP + FN}$ | Percentual geral de acertos (falha sob desbalanceamento). |
| **Precisão** | $\frac{TP}{TP + FP}$ | Confiabilidade do alerta positivo (custo alto de FP). |
| **Recall (Sensibilidade)** | $\frac{TP}{TP + FN}$ | Taxa de captura dos positivos reais (custo alto de FN). |
| **Especificidade (TNR)** | $\frac{TN}{TN + FP}$ | Taxa de proteção dos negativos reais. |
| **F1-Score** | $2 \cdot \frac{\text{Precisão} \cdot \text{Recall}}{\text{Precisão} + \text{Recall}}$ | Média harmônica padrão balanceada. |
| **F2-Score** | $\frac{5 \cdot \text{Precisão} \cdot \text{Recall}}{4 \cdot \text{Precisão} + \text{Recall}}$ | Dá $2\times$ mais peso ao Recall (foco em cobertura). |
| **F0.5-Score** | $\frac{1.25 \cdot \text{Precisão} \cdot \text{Recall}}{0.25 \cdot \text{Precisão} + \text{Recall}}$ | Dá $2\times$ mais peso à Precisão (foco em certeza). |


In [ ]:
# 1. Extração de probabilidades e predições no conjunto de validação
model_bin.eval()
val_logits_list, val_labels_list = [], []

with torch.no_grad():
    for batch_x, batch_y in val_loader_bin:
        batch_x = batch_x.to(device)
        logits = model_bin(batch_x)
        val_logits_list.extend(logits.cpu().numpy())
        val_labels_list.extend(batch_y.numpy())

val_probs_bin = 1 / (1 + np.exp(-np.array(val_logits_list)))
val_labels_bin = np.array(val_labels_list)
val_preds_bin = (val_probs_bin >= 0.50).astype(int)

# 2. Matriz de Confusão
cm_bin = confusion_matrix(val_labels_bin, val_preds_bin)
tn, fp, fn, tp = cm_bin.ravel()

# 3. Métricas
acc = accuracy_score(val_labels_bin, val_preds_bin)
prec = precision_score(val_labels_bin, val_preds_bin)
rec = recall_score(val_labels_bin, val_preds_bin)
spec = tn / (tn + fp)
f1 = f1_score(val_labels_bin, val_preds_bin)
f2 = fbeta_score(val_labels_bin, val_preds_bin, beta=2.0)
f05 = fbeta_score(val_labels_bin, val_preds_bin, beta=0.5)

# Visualização com Heatmap Anotado
plt.figure(figsize=(7, 5))
labels_annot = np.array([
    [f"TN (Verdadeiro Negativo)\n{tn}\n({tn/len(val_labels_bin):.1%})", f"FP (Falso Positivo)\n{fp}\n({fp/len(val_labels_bin):.1%})"],
    [f"FN (Falso Negativo)\n{fn}\n({fn/len(val_labels_bin):.1%})", f"TP (Verdadeiro Positivo)\n{tp}\n({tp/len(val_labels_bin):.1%})"]
])

sns.heatmap(cm_bin, annot=labels_annot, fmt="", cmap="Blues", cbar=False,
            xticklabels=['Previsto: 0 (Não Rec.)', 'Previsto: 1 (Recomendado)'],
            yticklabels=['Real: 0 (Não Rec.)', 'Real: 1 (Recomendado)'],
            annot_kws={"fontsize": 11})
plt.title("Matriz de Confusão Binária (Validação - Limiar = 0.50)", fontsize=12, fontweight='bold')
plt.show()

# Tabela Resumo Consolidada
metrics_summary_bin = pd.DataFrame({
    'Métrica': ['Acurácia', 'Precisão', 'Recall (Sensibilidade)', 'Especificidade (TNR)', 'F1-Score', 'F2-Score (Foco Recall)', 'F0.5-Score (Foco Precisão)'],
    'Valor': [acc, prec, rec, spec, f1, f2, f05],
    'Interpretação': [
        'Percentual geral de acertos',
        'Quando o modelo recomendou, quantos eram de fato recomendados',
        'De todos os recomendados reais, quantos o modelo capturou',
        'De todos os não recomendados, quantos o modelo identificou',
        'Média harmônica padrão equilibrada',
        'Ponderação com ênfase em cobertura (Recall)',
        'Ponderação com ênfase em certeza (Precisão)'
    ]
})
print("\n📊 Resumo Consolidado de Métricas Binárias:")
print(metrics_summary_bin.to_string(index=False, formatters={'Valor': '{:.4f}'.format}))


### 4.2 Comparativo de Curvas: ROC-AUC vs Precision-Recall (PR-AUC)

- **Curva ROC:** Mapeia $\text{Recall}$ contra $\text{FPR} = \frac{FP}{TN + FP}$. Quando a classe negativa ($TN$) é muito numerosa, o FPR permanece baixo, dando uma falsa sensação de perfeição.
- **Curva Precision-Recall (PR):** Mapeia $\text{Precisão}$ contra $\text{Recall}$. Como não depende de $TN$, ela expressa com clareza o desempenho do modelo na classe de interesse em cenários desbalanceados.


In [ ]:
# Curvas ROC e PR
fpr, tpr, _ = roc_curve(val_labels_bin, val_probs_bin)
roc_auc_val = auc(fpr, tpr)

prec_curve, rec_curve, _ = precision_recall_curve(val_labels_bin, val_probs_bin)
pr_auc_val = auc(rec_curve, prec_curve)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: ROC-AUC
axes[0].plot(fpr, tpr, color='#0284C7', lw=2.5, label=f'Curva ROC (AUC = {roc_auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', label='Baseline Aleatório (0.500)')
axes[0].set_title("Curva ROC (Sensível ao Volume de TNs)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Taxa de Falsos Positivos (FPR)")
axes[0].set_ylabel("Taxa de Verdadeiros Positivos (Recall / TPR)")
axes[0].legend(loc='lower right')

# Gráfico 2: PR-AUC
baseline_pr = val_labels_bin.mean()
axes[1].plot(rec_curve, prec_curve, color='#EA580C', lw=2.5, label=f'Curva PR (AUC = {pr_auc_val:.3f})')
axes[1].axhline(baseline_pr, color='#94A3B8', linestyle='--', label=f'Baseline Aleatório ({baseline_pr:.3f})')
axes[1].set_title("Curva Precision-Recall (Foco na Classe Positiva)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Recall (Taxa de Captura)")
axes[1].set_ylabel("Precisão (Confiabilidade dos Alarmes)")
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

print(f"📈 Área sob a Curva ROC (ROC-AUC): {roc_auc_val:.4f}")
print(f"🎯 Área sob a Curva PR (PR-AUC):   {pr_auc_val:.4f}")


### 4.3 Validação Cruzada Estratificada Binária (Stratified K-Fold) com PyTorch

1. **Preservação de Proporção:** O `StratifiedKFold` assegura que cada um dos 5 folds contenha exatamente a mesma proporção de ~82% recomendados e ~18% não-recomendados.
2. **Prevenção de Data Leakage no Loop:** Em cada iteração do fold, o vocabulário TF-IDF é ajustado **exclusivamente nos textos de treino do fold atual**.
3. **DataLoaders Dinâmicos:** Instanciamos os `DataLoader` de treino e validação a cada fold e treinamos uma rede inicializada do zero para obter a média ($\mu$) e desvio padrão ($\sigma$) das métricas.


In [ ]:
n_splits = 5
skf_bin = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_results_bin = []

print(f"🔄 Executando Validação Cruzada Estratificada Binária ({n_splits} Folds)...\n")

for fold, (train_idx, val_idx) in enumerate(skf_bin.split(X_tr_val_text, y_tr_val_bin), 1):
    # 1. Textos e Alvos brutos do Fold
    f_tr_text, f_val_text = X_tr_val_text[train_idx], X_tr_val_text[val_idx]
    f_y_tr, f_y_val = y_tr_val_bin[train_idx], y_tr_val_bin[val_idx]
    
    # 2. TF-IDF sem Data Leakage (Ajustado no Treino do Fold)
    f_tfidf = TfidfVectorizer(max_features=500, stop_words='english', sublinear_tf=True)
    f_X_tr = f_tfidf.fit_transform(f_tr_text).toarray()
    f_X_val = f_tfidf.transform(f_val_text).toarray()
    
    # 3. DataLoaders do Fold
    f_tr_loader = DataLoader(TensorDataset(torch.FloatTensor(f_X_tr), torch.FloatTensor(f_y_tr)), batch_size=64, shuffle=True)
    f_val_loader = DataLoader(TensorDataset(torch.FloatTensor(f_X_val), torch.FloatTensor(f_y_val)), batch_size=128, shuffle=False)
    
    # 4. Inicializar Modelo e Otimizador
    set_seed(42 + fold)
    f_model = BinaryReviewClassifier(f_X_tr.shape[1]).to(device)
    f_criterion = nn.BCEWithLogitsLoss()
    f_optimizer = optim.AdamW(f_model.parameters(), lr=1e-3, weight_decay=1e-2)
    
    # 5. Treinar
    for epoch in range(8):
        f_model.train()
        for bx, by in f_tr_loader:
            bx, by = bx.to(device), by.to(device)
            f_optimizer.zero_grad()
            logits = f_model(bx)
            loss = f_criterion(logits, by)
            loss.backward()
            f_optimizer.step()
            
    # 6. Avaliar no conjunto de validação do fold
    f_model.eval()
    f_logits, f_targets = [], []
    with torch.no_grad():
        for bx, by in f_val_loader:
            bx = bx.to(device)
            logits = f_model(bx)
            f_logits.extend(logits.cpu().numpy())
            f_targets.extend(by.numpy())
            
    f_probs = 1 / (1 + np.exp(-np.array(f_logits)))
    f_targets = np.array(f_targets)
    f_preds = (f_probs >= 0.50).astype(int)
    
    # Métricas do Fold
    f_acc = accuracy_score(f_y_val, f_preds)
    f_prec = precision_score(f_y_val, f_preds)
    f_rec = recall_score(f_y_val, f_preds)
    f_f1 = f1_score(f_y_val, f_preds)
    f_roc_auc = roc_auc_score(f_y_val, f_probs)
    prec_pts, rec_pts, _ = precision_recall_curve(f_y_val, f_probs)
    f_pr_auc = auc(rec_pts, prec_pts)
    
    cv_results_bin.append({
        'Fold': f'Fold {fold}',
        'Acurácia': f_acc,
        'Precisão': f_prec,
        'Recall': f_rec,
        'F1-Score': f_f1,
        'ROC-AUC': f_roc_auc,
        'PR-AUC': f_pr_auc
    })
    print(f"  ✅ Fold {fold}/{n_splits} | Acurácia: {f_acc:.4f} | Precisão: {f_prec:.4f} | Recall: {f_rec:.4f} | F1: {f_f1:.4f} | PR-AUC: {f_pr_auc:.4f}")

# Tabela Consolidada com Média e Desvio Padrão
df_cv_bin = pd.DataFrame(cv_results_bin)
mean_row_bin = df_cv_bin.mean(numeric_only=True).to_dict()
mean_row_bin['Fold'] = 'MÉDIA'
std_row_bin = df_cv_bin.std(numeric_only=True).to_dict()
std_row_bin['Fold'] = 'DESVIO PADRÃO'

df_cv_bin_summary = pd.concat([df_cv_bin, pd.DataFrame([mean_row_bin, std_row_bin])], ignore_index=True)
print("\n📋 Tabela Consolidada de Cross-Validation Binário (5 Folds):")
print(df_cv_bin_summary.to_string(index=False, formatters={
    'Acurácia': '{:.4f}'.format,
    'Precisão': '{:.4f}'.format,
    'Recall': '{:.4f}'.format,
    'F1-Score': '{:.4f}'.format,
    'ROC-AUC': '{:.4f}'.format,
    'PR-AUC': '{:.4f}'.format
}))


In [ ]:
# Gráfico de barras com médias e desvios padrão do Cross-Validation Binário
metrics_cols_bin = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC']
means_bin = [df_cv_bin[m].mean() for m in metrics_cols_bin]
stds_bin = [df_cv_bin[m].std() for m in metrics_cols_bin]

plt.figure(figsize=(10, 5))
bars = plt.bar(metrics_cols_bin, means_bin, yerr=stds_bin, capsize=6, color=['#0284C7', '#0D9488', '#16A34A', '#8B5CF6', '#F59E0B', '#EA580C'], alpha=0.85, edgecolor='black', linewidth=1.2)

plt.ylim(0.0, 1.05)
plt.title("Desempenho Médio do Modelo Binário com Stratified K-Fold (5 Folds)", fontsize=13, fontweight='bold')
plt.ylabel("Score")

for bar, mean_val, std_val in zip(bars, means_bin, stds_bin):
    y_pos = bar.get_height() + 0.04
    plt.text(bar.get_x() + bar.get_width()/2, y_pos, f"{mean_val:.3f}\n(±{std_val:.3f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. PARTE 2 — Classificação Multiclasse e Avaliação de Dados Ordinais (`Rating` de 1 a 5 Estrelas)

Agora a rede neural deve prever a **nota exata em estrelas ($1, 2, 3, 4 \text{ ou } 5$)** com base apenas no texto da avaliação.

### 📐 A Natureza Ordinal do Rating e os Desafios de Avaliação:
1. **O Rating é Ordinal, não Nominal:** Existe uma escala com ordem e magnitude ($1 < 2 < 3 < 4 < 5$). Errar de 5 para 4 estrelas é muito menos grave do que errar de 5 para 1 estrela.
2. **Matriz de Confusão 5x5:** A diagonal principal representa os acertos exatos. Em problemas ordinais, inspecionamos se os erros se concentram nas **diagonais adjacentes** ($\pm 1$ estrela de erro).
3. **Métricas Clássicas vs Métricas Ordinais:**
   - **Macro-F1:** Avalia o aprendizado equilibrado em todas as notas (incluindo as notas raras de 1 e 2 estrelas).
   - **MAE nas Estrelas ($\text{Mean Absolute Error}$):** Mede em média quantas estrelas de erro o modelo comete: $\text{MAE} = \frac{1}{N} \sum |y_{\text{real}} - y_{\text{pred}}|$.
   - **Quadratic Weighted Kappa (QWK):** Métrica padrão da indústria para avaliações e ratings ordinais. O QWK varia de 0 a 1 e penaliza desvios quadraticamente $(y_{\text{real}} - y_{\text{pred}})^2$.


In [ ]:
class MultiClassReviewClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=5, hidden_dim=128, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, num_classes) # Saída de 5 logits lineares
        )

    def forward(self, x):
        return self.net(x)

# Instanciar e treinar modelo multiclasse
set_seed(42)
model_multi = MultiClassReviewClassifier(input_dim, num_classes=5).to(device)
criterion_multi = nn.CrossEntropyLoss()
optimizer_multi = optim.AdamW(model_multi.parameters(), lr=1e-3, weight_decay=1e-2)

epochs = 8
print("🔄 Treinando Modelo Multiclasse (5 Classes de Estrelas)...")
for epoch in range(1, epochs + 1):
    model_multi.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader_multi:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer_multi.zero_grad()
        logits = model_multi(batch_x)
        loss = criterion_multi(logits, batch_y)
        loss.backward()
        optimizer_multi.step()
        total_loss += loss.item() * len(batch_y)
    
    avg_loss = total_loss / len(train_loader_multi.dataset)
    print(f"  • Época {epoch:02d}/{epochs:02d} | Loss Médio de Treino (Cross-Entropy): {avg_loss:.4f}")

print("✅ Treinamento Multiclasse Concluído!")


### 5.1 Matriz de Confusão 5x5, Relatório e Métricas Ordinais (MAE e QWK)

Visualizamos o heatmap 5x5 e calculamos tanto as métricas categóricas tradicionais (**Acurácia**, **Macro F1**, **Weighted F1**) quanto as métricas ordinais fundamentais (**MAE em Estrelas** e **Quadratic Weighted Kappa - QWK**).


In [ ]:
# 1. Extração de predições no conjunto de validação
model_multi.eval()
val_preds_multi_list, val_labels_multi_list = [], []

with torch.no_grad():
    for batch_x, batch_y in val_loader_multi:
        batch_x = batch_x.to(device)
        logits = model_multi(batch_x)
        preds = torch.argmax(logits, dim=-1)
        val_preds_multi_list.extend(preds.cpu().numpy())
        val_labels_multi_list.extend(batch_y.numpy())

val_preds_multi = np.array(val_preds_multi_list)
val_labels_multi = np.array(val_labels_multi_list)

# 2. Matriz de Confusão 5x5
star_labels = ['1 Estrela', '2 Estrelas', '3 Estrelas', '4 Estrelas', '5 Estrelas']
cm_multi = confusion_matrix(val_labels_multi, val_preds_multi)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_multi, annot=True, fmt="d", cmap="Purples", cbar=True,
            xticklabels=star_labels, yticklabels=star_labels)
plt.title("Matriz de Confusão Multiclasse (Validação - 1 a 5 Estrelas)", fontsize=12, fontweight='bold')
plt.xlabel("Classe Prevista")
plt.ylabel("Classe Real")
plt.show()

# 3. Métricas Globais (Categóricas e Ordinais)
acc_multi = accuracy_score(val_labels_multi, val_preds_multi)
macro_f1 = f1_score(val_labels_multi, val_preds_multi, average='macro')
weighted_f1 = f1_score(val_labels_multi, val_preds_multi, average='weighted')

# Métricas Ordinais (convertendo de 0..4 para 1..5 estrelas reais)
mae_stars = mean_absolute_error(val_labels_multi + 1, val_preds_multi + 1)
qwk_score = cohen_kappa_score(val_labels_multi, val_preds_multi, weights='quadratic')

print("📊 Métricas de Avaliação Multiclasse e Ordinal:")
print(f"  • Acurácia Exata:             {acc_multi:.4f} ({acc_multi*100:.2f}%)")
print(f"  • Macro F1-Score:             {macro_f1:.4f} (Igual peso para todas as notas)")
print(f"  • Weighted F1-Score:          {weighted_f1:.4f} (Ponderado pelo volume)")
print(f"  • MAE em Estrelas:            {mae_stars:.4f} estrelas (Erro médio absoluto de nota)")
print(f"  • Quadratic Weighted Kappa:   {qwk_score:.4f} (Concordância ordinal penalizando distância quadrática)")

print("\n📋 Relatório de Classificação Detalhado por Nota de Estrela:")
print(classification_report(val_labels_multi, val_preds_multi, target_names=star_labels, digits=4))


### 5.2 Validação Cruzada Estratificada Multiclasse (Stratified K-Fold) com PyTorch

Aplicamos o `StratifiedKFold` estratificado pelas **5 classes de notas (`Rating`)**, computando para cada partição tanto as métricas categóricas (Acurácia, Macro-F1, Weighted-F1) quanto as métricas ordinais (MAE e QWK).


In [ ]:
skf_multi = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
cv_results_multi = []

print(f"🔄 Executando Validação Cruzada Estratificada Multiclasse ({n_splits} Folds)...\n")

for fold, (train_idx, val_idx) in enumerate(skf_multi.split(X_tr_val_text, y_tr_val_multi), 1):
    # 1. Textos e Alvos brutos do Fold
    f_tr_text, f_val_text = X_tr_val_text[train_idx], X_tr_val_text[val_idx]
    f_y_tr, f_y_val = y_tr_val_multi[train_idx], y_tr_val_multi[val_idx]
    
    # 2. TF-IDF sem Data Leakage (Ajustado no Treino do Fold)
    f_tfidf = TfidfVectorizer(max_features=500, stop_words='english', sublinear_tf=True)
    f_X_tr = f_tfidf.fit_transform(f_tr_text).toarray()
    f_X_val = f_tfidf.transform(f_val_text).toarray()
    
    # 3. DataLoaders do Fold (LongTensor para os alvos multiclasse)
    f_tr_loader = DataLoader(TensorDataset(torch.FloatTensor(f_X_tr), torch.LongTensor(f_y_tr)), batch_size=64, shuffle=True)
    f_val_loader = DataLoader(TensorDataset(torch.FloatTensor(f_X_val), torch.LongTensor(f_y_val)), batch_size=128, shuffle=False)
    
    # 4. Inicializar Modelo e Otimizador
    set_seed(42 + fold)
    f_model = MultiClassReviewClassifier(f_X_tr.shape[1], num_classes=5).to(device)
    f_criterion = nn.CrossEntropyLoss()
    f_optimizer = optim.AdamW(f_model.parameters(), lr=1e-3, weight_decay=1e-2)
    
    # 5. Treinar
    for epoch in range(8):
        f_model.train()
        for bx, by in f_tr_loader:
            bx, by = bx.to(device), by.to(device)
            f_optimizer.zero_grad()
            logits = f_model(bx)
            loss = f_criterion(logits, by)
            loss.backward()
            f_optimizer.step()
            
    # 6. Avaliar
    f_model.eval()
    f_preds, f_targets = [], []
    with torch.no_grad():
        for bx, by in f_val_loader:
            bx = bx.to(device)
            logits = f_model(bx)
            preds = torch.argmax(logits, dim=-1)
            f_preds.extend(preds.cpu().numpy())
            f_targets.extend(by.numpy())
            
    f_preds = np.array(f_preds)
    f_targets = np.array(f_targets)
    
    # Métricas Categóricas e Ordinais
    f_acc = accuracy_score(f_targets, f_preds)
    f_macro_f1 = f1_score(f_targets, f_preds, average='macro')
    f_weighted_f1 = f1_score(f_targets, f_preds, average='weighted')
    f_mae = mean_absolute_error(f_targets + 1, f_preds + 1)
    f_qwk = cohen_kappa_score(f_targets, f_preds, weights='quadratic')
    
    cv_results_multi.append({
        'Fold': f'Fold {fold}',
        'Acurácia': f_acc,
        'Macro F1': f_macro_f1,
        'Weighted F1': f_weighted_f1,
        'MAE (Estrelas)': f_mae,
        'QWK': f_qwk
    })
    print(f"  ✅ Fold {fold}/{n_splits} | Acurácia: {f_acc:.4f} | Macro F1: {f_macro_f1:.4f} | MAE: {f_mae:.3f}★ | QWK: {f_qwk:.4f}")

# Tabela Consolidada Multiclasse
df_cv_multi = pd.DataFrame(cv_results_multi)
mean_row_multi = df_cv_multi.mean(numeric_only=True).to_dict()
mean_row_multi['Fold'] = 'MÉDIA'
std_row_multi = df_cv_multi.std(numeric_only=True).to_dict()
std_row_multi['Fold'] = 'DESVIO PADRÃO'

df_cv_multi_summary = pd.concat([df_cv_multi, pd.DataFrame([mean_row_multi, std_row_multi])], ignore_index=True)
print("\n📋 Tabela Consolidada de Cross-Validation Multiclasse / Ordinal (5 Folds):")
print(df_cv_multi_summary.to_string(index=False, formatters={
    'Acurácia': '{:.4f}'.format,
    'Macro F1': '{:.4f}'.format,
    'Weighted F1': '{:.4f}'.format,
    'MAE (Estrelas)': '{:.4f}'.format,
    'QWK': '{:.4f}'.format
}))


In [ ]:
# Gráficos de barras com médias e desvios padrão do Cross-Validation Multiclasse e Ordinal
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: Métricas Categóricas (Acurácia, Macro-F1, Weighted-F1, QWK)
cat_cols = ['Acurácia', 'Macro F1', 'Weighted F1', 'QWK']
cat_means = [df_cv_multi[m].mean() for m in cat_cols]
cat_stds = [df_cv_multi[m].std() for m in cat_cols]

bars1 = axes[0].bar(cat_cols, cat_means, yerr=cat_stds, capsize=6, color=['#7C3AED', '#EC4899', '#3B82F6', '#10B981'], alpha=0.85, edgecolor='black', linewidth=1.2)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title("Métricas de Concordância e F1 (5 Folds)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Score")

for bar, mean_val, std_val in zip(bars1, cat_means, cat_stds):
    y_pos = bar.get_height() + 0.04
    axes[0].text(bar.get_x() + bar.get_width()/2, y_pos, f"{mean_val:.3f}\n(±{std_val:.3f})", ha='center', va='bottom', fontsize=9, fontweight='bold')

# Gráfico 2: Erro Médio Absoluto em Estrelas (MAE)
mae_mean = df_cv_multi['MAE (Estrelas)'].mean()
mae_std = df_cv_multi['MAE (Estrelas)'].std()

bars2 = axes[1].bar(['MAE (Estrelas)'], [mae_mean], yerr=[mae_std], capsize=6, color=['#F59E0B'], alpha=0.85, edgecolor='black', linewidth=1.2, width=0.4)
axes[1].set_ylim(0.0, 1.5)
axes[1].set_title("Erro Médio Absoluto em Estrelas (Menor é Melhor)", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Estrelas de Erro Médio")

axes[1].text(bars2[0].get_x() + bars2[0].get_width()/2, mae_mean + 0.06, f"{mae_mean:.3f}★\n(±{mae_std:.3f})", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 6. PARTE 3 — Avaliação Final no Conjunto de Teste Cego

Avaliamos os dois modelos finais no conjunto de **Teste Cego (Holdout)**, garantindo que ambos generalizam bem em dados nunca antes vistos.


In [ ]:
# 1. Teste Cego no Modelo Binário
model_bin.eval()
test_logits_bin, test_labels_bin = [], []
with torch.no_grad():
    for bx, by in test_loader_bin:
        bx = bx.to(device)
        test_logits_bin.extend(model_bin(bx).cpu().numpy())
        test_labels_bin.extend(by.numpy())

test_probs_bin = 1 / (1 + np.exp(-np.array(test_logits_bin)))
test_preds_bin = (test_probs_bin >= 0.50).astype(int)

print("🎯 [CENÁRIO 1] Relatório de Teste Cego - Classificação Binária (Recomendação):")
print(classification_report(test_labels_bin, test_preds_bin, target_names=['Não Recomendado (0)', 'Recomendado (1)'], digits=4))

# 2. Teste Cego no Modelo Multiclasse
model_multi.eval()
test_preds_multi, test_labels_multi = [], []
with torch.no_grad():
    for bx, by in test_loader_multi:
        bx = bx.to(device)
        preds = torch.argmax(model_multi(bx), dim=-1)
        test_preds_multi.extend(preds.cpu().numpy())
        test_labels_multi.extend(by.numpy())

test_preds_multi = np.array(test_preds_multi)
test_labels_multi = np.array(test_labels_multi)

test_mae = mean_absolute_error(test_labels_multi + 1, test_preds_multi + 1)
test_qwk = cohen_kappa_score(test_labels_multi, test_preds_multi, weights='quadratic')

print(f"\n🎯 [CENÁRIO 2] Relatório de Teste Cego - Classificação Multiclasse (1 a 5 Estrelas):")
print(f"  • MAE Teste: {test_mae:.4f} estrelas | QWK Teste: {test_qwk:.4f}")
print(classification_report(test_labels_multi, test_preds_multi, target_names=star_labels, digits=4))


## 7. Síntese e Guia de Melhores Práticas

### 📌 Resumo dos Princípios de Avaliação no PyTorch:
1. **Blindagem contra Data Leakage:**
   - **Target Leakage:** Não use variáveis que entregam o alvo diretamente (ex: `Rating` para prever recomendação).
   - **Temporal Leakage:** Não use variáveis criadas a posteriori pelo sistema ou por outros usuários (ex: `Positive Feedback Count`).
2. **Avaliação Binária em Classes Desbalanceadas:**
   - Acurácia pode ser ilusória quando uma classe domina (~82%).
   - Monitore a **Curva PR-AUC** e o **$F_1$-Score**.
   - Use $F_2$-Score quando perder positivos for crítico e $F_{0.5}$-Score quando falsos alarmes forem caros.
3. **Avaliação de Dados Ordinais (Ratings/Notas):**
   - Ao formular como multiclasse, inspecione a dispersão da **Matriz de Confusão 5x5** em torno da diagonal.
   - Acompanhe o **Macro $F_1$-Score** para vigiar o aprendizado nas classes minoritárias (1 e 2 estrelas).
   - Calcule o **MAE em Estrelas** e o **Quadratic Weighted Kappa (QWK)** para medir o impacto da distância das previsões.
4. **Stratified K-Fold + PyTorch DataLoader:**
   - O `StratifiedKFold` garante partições equilibradas tanto no cenário binário quanto no multiclasse.
   - Ajustar o pré-processamento (TF-IDF) **dentro de cada fold** assegura uma estimativa 100% livre de vazamento de dados.
